# Batch Compare Triangle-Field Flow Against Previous Gaussian Flows

This notebook compares the triangle-field flow model against each previous Gaussian-distance flow model on the same set of meshes.

For each mesh and each previous model it saves two side-by-side images:

- `d_tri` from the triangle model next to Gaussian `edge RGB`
- `d_vert` from the triangle model next to Gaussian `vertex RGB`

It uses one fixed camera view per method. The triangle scalar fields are rendered with configurable colormaps instead of grayscale.

In [1]:
from pathlib import Path

ROOT = Path('/nfs/turbo/coe-jjparkcv-medium/koussa/neuframe')
SPLIT = 'test'

# Select 64 meshes by index from the triangle reference dataset, then match by SHA in every Gaussian dataset.
START_INDEX = 0
NUM_MESHES = 64
TARGET_SHA256S = None  # Optional explicit list of SHA strings. Overrides START_INDEX/NUM_MESHES.

# These are checkpoint steps, not Euler sampling steps.
TRIANGLE_CKPT = 40000
GAUSSIAN_CKPT = 60000
EMA_RATE = None

SAMPLING_STEPS = 12
GUIDANCE_STRENGTH = 1.0
RENDER_RESOLUTION = 512
SEED = 0

# Triangle scalar-field colormaps. Valid matplotlib colormap names work here.
D_TRI_COLORMAP = 'magma'
D_VERT_COLORMAP = 'viridis'

TRIANGLE_PRESET = {
    'label': 'Triangle field + shape latent',
    'target_kind': 'triangle',
    'run_dir': ROOT / 'outputs' / 'michelangelo_shape2triangle_field_flow_51483691_step0060000',
    'dataset_name': 'MichelangeloShapeConditionedTriangleFieldSLat',
    'triangle_field_latent_name': 'triangle_field_vae_51483691_step0060000_256',
    'michelangelo_latent_name': 'shapevae256_pretrained',
    'shape_latent_name': 'occupancy_shape_vae_step0110000_256',
    'ckpt': TRIANGLE_CKPT,
}

GAUSSIAN_PRESETS = {
    'gaussian_shape_kl5e3': {
        'label': 'Gaussian KL5e-3 + shape latent',
        'target_kind': 'gaussian',
        'run_dir': ROOT / 'outputs' / 'michelangelo_shape2gaussian_distance_flow_kl5e3',
        'dataset_name': 'MichelangeloShapeConditionedGaussianDistanceSLat',
        'gaussian_distance_latent_name': 'gaussian_distance_vae_kl5e3_step0310000_256',
        'michelangelo_latent_name': 'shapevae256_pretrained',
        'shape_latent_name': 'occupancy_shape_vae_step0110000_256',
        'ckpt': GAUSSIAN_CKPT,
    },
    'gaussian_michelangelo_kl5e3': {
        'label': 'Gaussian KL5e-3 + Michelangelo only',
        'target_kind': 'gaussian',
        'run_dir': ROOT / 'outputs' / 'michelangelo2gaussian_distance_flow_kl5e3',
        'dataset_name': 'MichelangeloConditionedGaussianDistanceSLat',
        'gaussian_distance_latent_name': 'gaussian_distance_vae_kl5e3_step0310000_256',
        'michelangelo_latent_name': 'shapevae256_pretrained',
        'shape_latent_name': None,
        'ckpt': GAUSSIAN_CKPT,
    },
    'gaussian_michelangelo_base': {
        'label': 'Gaussian base + Michelangelo only',
        'target_kind': 'gaussian',
        'run_dir': ROOT / 'outputs' / 'michelangelo2gaussian_distance_flow_49678820',
        'dataset_name': 'MichelangeloConditionedGaussianDistanceSLat',
        'gaussian_distance_latent_name': 'gaussian_distance_vae_step0350000_256',
        'michelangelo_latent_name': 'shapevae256_pretrained',
        'shape_latent_name': None,
        'ckpt': GAUSSIAN_CKPT,
    },
}

ENABLED_GAUSSIAN_PRESETS = [
    'gaussian_shape_kl5e3',
    'gaussian_michelangelo_kl5e3',
    'gaussian_michelangelo_base',
]

OUTPUT_DIR = (
    ROOT / 'outputs' / 'flow_generation_comparisons' /
    f'{SPLIT}_triangle_step{TRIANGLE_CKPT:07d}_gaussian_step{GAUSSIAN_CKPT:07d}_n{NUM_MESHES}'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUTPUT_DIR)

Output directory: /nfs/turbo/coe-jjparkcv-medium/koussa/neuframe/outputs/flow_generation_comparisons/test_triangle_step0040000_gaussian_step0060000_n64


## Load Utilities

In [2]:
import copy
import gc
import json
import sys
from dataclasses import dataclass

import matplotlib.cm as cm
import numpy as np
import torch
from IPython.display import display
from PIL import Image, ImageDraw

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'trellis2').exists():
    REPO_ROOT = Path('/home/koussa/scratch/TRELLIS.2')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from eval_gaussian_distance_flow import build_data_dir as build_gaussian_data_dir
from eval_gaussian_distance_flow import load_denoiser_checkpoint
from eval_triangle_field_flow import build_data_dir as build_triangle_data_dir
from trellis2 import datasets, models
from trellis2.pipelines.samplers import FlowEulerCfgSampler
from trellis2.renderers import VoxelRenderer
from trellis2.representations import Voxel
from trellis2.utils.data_utils import recursive_to_device
import utils3d

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

@dataclass
class LoadedFlow:
    name: str
    preset: dict
    cfg: dict
    dataset: object
    denoiser: torch.nn.Module
    ckpt_path: str


def normalization_path(latent_dir, latent_name):
    return ROOT / 'splits' / 'train' / latent_dir / latent_name / 'normalization.json'


def build_data_dir(split, preset):
    if preset['target_kind'] == 'triangle':
        return build_triangle_data_dir(
            ROOT,
            split,
            preset['triangle_field_latent_name'],
            preset['michelangelo_latent_name'],
            preset['shape_latent_name'],
        )
    return build_gaussian_data_dir(
        ROOT,
        split,
        preset['gaussian_distance_latent_name'],
        preset['michelangelo_latent_name'],
        preset['shape_latent_name'],
    )


def load_flow(name, preset):
    run_dir = Path(preset['run_dir'])
    cfg = json.load(open(run_dir / 'config.json', 'r'))
    if cfg['dataset']['name'] != preset['dataset_name']:
        raise ValueError(f"{name}: preset dataset {preset['dataset_name']} != config dataset {cfg['dataset']['name']}")

    dataset_args = copy.deepcopy(cfg['dataset']['args'])
    dataset_args['snapshot_render_resolution'] = RENDER_RESOLUTION
    if preset['target_kind'] == 'triangle':
        norm_path = normalization_path('triangle_field_latents', preset['triangle_field_latent_name'])
        dataset_args['triangle_field_slat_normalization_path'] = str(norm_path)
    else:
        norm_path = normalization_path('gaussian_distance_latents', preset['gaussian_distance_latent_name'])
        dataset_args['gaussian_distance_slat_normalization_path'] = str(norm_path)
    if not norm_path.exists():
        raise FileNotFoundError(f'{name}: missing target latent normalization: {norm_path}')

    if preset.get('shape_latent_name') is not None:
        shape_norm = normalization_path('shape_latents', preset['shape_latent_name'])
        dataset_args['shape_slat_normalization_path'] = str(shape_norm)
        if not shape_norm.exists():
            raise FileNotFoundError(f'{name}: missing shape latent normalization: {shape_norm}')

    dataset = getattr(datasets, cfg['dataset']['name'])(json.dumps(build_data_dir(SPLIT, preset)), **dataset_args)

    denoiser_cfg = cfg['models']['denoiser']
    denoiser = getattr(models, denoiser_cfg['name'])(**denoiser_cfg['args']).cuda().eval()
    ckpt_path = load_denoiser_checkpoint(
        denoiser,
        run_dir,
        int(preset['ckpt']),
        ema_rate=EMA_RATE,
        device=torch.device('cuda'),
    )
    print(f'Loaded {name}: {preset["label"]}')
    print('  checkpoint:', ckpt_path)
    return LoadedFlow(name, preset, cfg, dataset, denoiser, ckpt_path)


def unload_flow(flow):
    if flow is None:
        return
    if flow.preset['target_kind'] == 'triangle':
        if getattr(flow.dataset, 'triangle_field_slat_dec', None) is not None:
            flow.dataset._delete_triangle_field_slat_dec()
    else:
        if getattr(flow.dataset, 'gaussian_distance_slat_dec', None) is not None:
            flow.dataset._delete_gaussian_distance_slat_dec()
    del flow.denoiser
    gc.collect()
    torch.cuda.empty_cache()

## Camera, Sampling, Decode, and Render Helpers

In [3]:
renderer = VoxelRenderer()
renderer.rendering_options.resolution = RENDER_RESOLUTION
renderer.rendering_options.ssaa = 4

# One deterministic camera per sample SHA. The same camera is reused for every method on that sample.
def camera_for_sha(sha256):
    seed = int(sha256[:16], 16) ^ int(SEED)
    rng = np.random.default_rng(seed)
    yaw = rng.uniform(0.0, 2.0 * np.pi)
    pitch = rng.uniform(np.deg2rad(-20.0), np.deg2rad(35.0))
    origin = torch.tensor([
        np.sin(yaw) * np.cos(pitch),
        np.cos(yaw) * np.cos(pitch),
        np.sin(pitch),
    ], dtype=torch.float32, device='cuda') * 2
    fov = torch.deg2rad(torch.tensor(30.0, device='cuda'))
    extrinsic = utils3d.torch.extrinsics_look_at(
        origin,
        torch.tensor([0, 0, 0], dtype=torch.float32, device='cuda'),
        torch.tensor([0, 0, 1], dtype=torch.float32, device='cuda'),
    )
    intrinsic = utils3d.torch.intrinsics_from_fov_xy(fov, fov)
    return extrinsic, intrinsic


def find_instance(dataset, sha256):
    for idx, (root_info, candidate_sha) in enumerate(dataset.instances):
        if candidate_sha == sha256:
            return idx, root_info
    raise KeyError(f'{sha256} not found in {dataset.__class__.__name__}')


def get_batch(flow, sha256):
    _, root_info = find_instance(flow.dataset, sha256)
    item = flow.dataset.get_instance(root_info, sha256)
    return recursive_to_device(flow.dataset.collate_fn([item]), 'cuda')


def sample_flow(flow, batch, seed_offset=0):
    x_0 = batch['x_0']
    torch.manual_seed(SEED + seed_offset)
    torch.cuda.manual_seed_all(SEED + seed_offset)
    noise = x_0.replace(torch.randn_like(x_0.feats))
    kwargs = {'cond': batch['cond'], 'neg_cond': batch['neg_cond']}
    if 'concat_cond' in batch:
        kwargs['concat_cond'] = batch['concat_cond']
    sampler = FlowEulerCfgSampler(flow.cfg['trainer']['args']['sigma_min'])
    with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        return sampler.sample(
            flow.denoiser,
            noise=noise,
            steps=SAMPLING_STEPS,
            guidance_strength=GUIDANCE_STRENGTH,
            verbose=False,
            **kwargs,
        ).samples


def unnormalize_shape_z(dataset, shape_z):
    if shape_z is not None and getattr(dataset, 'shape_slat_normalization', None) is not None:
        return shape_z.replace(shape_z.feats * dataset.shape_slat_std.to(shape_z.device) + dataset.shape_slat_mean.to(shape_z.device))
    return shape_z


def decode_sample(flow, z, batch):
    dataset = flow.dataset
    z = z.cuda()
    shape_z = batch.get('concat_cond', None)

    if flow.preset['target_kind'] == 'triangle':
        dataset._loading_triangle_field_slat_dec()
        if dataset.triangle_field_slat_normalization is not None:
            z = z.replace(z.feats * dataset.triangle_field_slat_std.to(z.device) + dataset.triangle_field_slat_mean.to(z.device))
        shape_z = unnormalize_shape_z(dataset, shape_z.cuda() if shape_z is not None else None)
        if shape_z is not None:
            _, subs = dataset.shape_slat_dec(shape_z, return_subs=True)
            return dataset.triangle_field_slat_dec(z, guide_subs=subs)[0]
        cache = dataset._load_triangle_field_slat_cache(batch['triangle_field_slat_cache_path'][0], z.device)
        z._scale = cache['scale']
        z._spatial_cache = cache['spatial_cache']
        return dataset.triangle_field_slat_dec(z)[0]

    dataset._loading_gaussian_distance_slat_dec()
    if dataset.gaussian_distance_slat_normalization is not None:
        z = z.replace(z.feats * dataset.gaussian_distance_slat_std.to(z.device) + dataset.gaussian_distance_slat_mean.to(z.device))
    shape_z = unnormalize_shape_z(dataset, shape_z.cuda() if shape_z is not None else None)
    if shape_z is not None:
        _, subs = dataset.shape_slat_dec(shape_z, return_subs=True)
        return dataset.gaussian_distance_slat_dec(z, guide_subs=subs)[0]
    cache = dataset._load_gaussian_distance_slat_cache(batch['gaussian_distance_slat_cache_path'][0], z.device)
    z._scale = cache['scale']
    z._spatial_cache = cache['spatial_cache']
    return dataset.gaussian_distance_slat_dec(z)[0]


def apply_colormap(values, cmap_name):
    values_np = values.detach().float().cpu().numpy().reshape(-1).clip(0, 1)
    rgb = cm.get_cmap(cmap_name)(values_np)[:, :3]
    return torch.from_numpy(rgb).to(values.device, dtype=torch.float32)


def render_attrs(voxel, attrs, resolution, sha256):
    rep = Voxel(
        origin=[-0.5, -0.5, -0.5],
        voxel_size=1 / resolution,
        coords=voxel.coords[:, 1:].contiguous(),
        attrs=None,
        layout={'color': slice(0, 3)},
    )
    extrinsic, intrinsic = camera_for_sha(sha256)
    with torch.autocast(device_type='cuda', enabled=False):
        image = renderer.render(rep, extrinsic.float(), intrinsic.float(), colors_overwrite=attrs.float().clamp(0, 1))['color'].float()
    image = image.detach().cpu().permute(1, 2, 0).numpy()
    return Image.fromarray((np.clip(image, 0, 1) * 255).astype(np.uint8))


def add_header(image, text):
    image = image.copy()
    draw = ImageDraw.Draw(image)
    draw.rectangle((0, 0, image.width, 32), fill=(0, 0, 0))
    draw.text((8, 9), text, fill=(255, 255, 255))
    return image


def save_pair(left, right, output_path):
    canvas = Image.new('RGB', (left.width + right.width, max(left.height, right.height)), color=(0, 0, 0))
    canvas.paste(left, (0, 0))
    canvas.paste(right, (left.width, 0))
    canvas.save(output_path)
    return canvas

print('Helpers ready.')

Helpers ready.


## Generate Triangle Reference Images for 64 Meshes

In [4]:
triangle_flow = load_flow('triangle_shape', TRIANGLE_PRESET)

if TARGET_SHA256S is None:
    selected_sha256s = [sha for _, sha in triangle_flow.dataset.instances[START_INDEX:START_INDEX + NUM_MESHES]]
else:
    selected_sha256s = list(TARGET_SHA256S)

triangle_images = {}
triangle_meta = {}

for mesh_i, sha256 in enumerate(selected_sha256s):
    print(f'[{mesh_i + 1}/{len(selected_sha256s)}] triangle {sha256}')
    batch = get_batch(triangle_flow, sha256)
    z = sample_flow(triangle_flow, batch, seed_offset=mesh_i)
    voxel = decode_sample(triangle_flow, z, batch)
    feats = voxel.feats.float()
    if getattr(triangle_flow.dataset, 'triangle_field_distance_transform', 'none') == 'minus_one_one':
        feats = feats * 0.5 + 0.5
    d_tri_img = add_header(
        render_attrs(voxel, apply_colormap(feats[:, 0:1], D_TRI_COLORMAP), triangle_flow.dataset.resolution, sha256),
        f'triangle d_tri ({D_TRI_COLORMAP})',
    )
    d_vert_img = add_header(
        render_attrs(voxel, apply_colormap(feats[:, 1:2], D_VERT_COLORMAP), triangle_flow.dataset.resolution, sha256),
        f'triangle d_vert ({D_VERT_COLORMAP})',
    )
    triangle_images[sha256] = {'d_tri': d_tri_img, 'd_vert': d_vert_img}
    triangle_meta[sha256] = {'tokens': int(z.feats.shape[0])}

unload_flow(triangle_flow)
print('Triangle images cached in memory:', len(triangle_images))

[SPARSE] Conv backend: flex_gemm; Attention backend: flash_attn
[ATTENTION] Using backend: flash_attn
Loaded triangle_shape: Triangle field + shape latent
  checkpoint: /nfs/turbo/coe-jjparkcv-medium/koussa/neuframe/outputs/michelangelo_shape2triangle_field_flow_51483691_step0060000/ckpts/denoiser_step0040000.pt
[1/63] triangle 0083b931abca7ba79abbb9e7ef851633297d0860436fd677d2279927867bbd6d


/tmp/ipykernel_3352067/341683471.py:98: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgb = cm.get_cmap(cmap_name)(values_np)[:, :3]
/tmp/ipykernel_3352067/341683471.py:98: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  rgb = cm.get_cmap(cmap_name)(values_np)[:, :3]


[2/63] triangle 020452235d4c2b085455c6d34afa86b15c0d9f737bda5fb90d944dc0f285e640
[3/63] triangle 035e8e5f3fb4f1103d1ab67456283f4d038a8537f5d25454843594fdd90d1f53
[4/63] triangle 049db641b0942cfca4f3bc0039a913276c21bc9e42cecf52e1a971b15d9e9763
[5/63] triangle 0b008fcdc842a59acee79b24a104cd688b0669ca9392d296c28c83f12fc73e0b
[6/63] triangle 0b7ce9b1d6bb838deb7936aae55dbbedc045c0bdb8f7b2363d2bcc38dc95b0dd
[7/63] triangle 0e49c47a6e29c2333c35e4741f30a9920904d147b3919213a46747e0f3c79292
[8/63] triangle 1283d3ddfcd2a7917b780c3200aa7c64574ed5682c2719b42fe6981a0daa2ba5
[9/63] triangle 1a019fe0926f673da89544b94dbdb474377e7bf6b61a4caa398f17b3c020dd81
[10/63] triangle 1a18a03fe1497ad902f3f84020af1115de554fd8a91b09a49ec2b1c8a6276ccd
[11/63] triangle 1ca7999fac9765f28ff80481e1fc90f3215b8c48dc968e3c092ddf873a3ac258
[12/63] triangle 1e5296884224b99dda4588505c3a5e40207f83f53cdf31641c2be6a3e95ff27c
[13/63] triangle 216843654f0a292f32af07971d656b067325f699ef7409ebd3980a64f2403939
[14/63] triangle 22549a6

## Compare Triangle Against Each Previous Gaussian Flow

In [5]:
summary = []

for gaussian_name in ENABLED_GAUSSIAN_PRESETS:
    preset = GAUSSIAN_PRESETS[gaussian_name]
    gaussian_flow = load_flow(gaussian_name, preset)

    edge_dir = OUTPUT_DIR / gaussian_name / 'd_tri_vs_edge_rgb'
    vert_dir = OUTPUT_DIR / gaussian_name / 'd_vert_vs_vertex_rgb'
    edge_dir.mkdir(parents=True, exist_ok=True)
    vert_dir.mkdir(parents=True, exist_ok=True)

    for mesh_i, sha256 in enumerate(selected_sha256s):
        print(f'[{gaussian_name}] [{mesh_i + 1}/{len(selected_sha256s)}] {sha256}')
        batch = get_batch(gaussian_flow, sha256)
        z = sample_flow(gaussian_flow, batch, seed_offset=mesh_i)
        voxel = decode_sample(gaussian_flow, z, batch)
        feats = (voxel.feats.float() * 0.5 + 0.5).clamp(0, 1)

        edge_img = add_header(
            render_attrs(voxel, feats[:, 0:3], gaussian_flow.dataset.resolution, sha256),
            f'{preset["label"]} edge RGB',
        )
        vertex_img = add_header(
            render_attrs(voxel, feats[:, 3:6], gaussian_flow.dataset.resolution, sha256),
            f'{preset["label"]} vertex RGB',
        )

        edge_path = edge_dir / f'{mesh_i:03d}_{sha256}.png'
        vert_path = vert_dir / f'{mesh_i:03d}_{sha256}.png'
        save_pair(triangle_images[sha256]['d_tri'], edge_img, edge_path)
        save_pair(triangle_images[sha256]['d_vert'], vertex_img, vert_path)
        summary.append({
            'comparison': gaussian_name,
            'index': mesh_i,
            'sha256': sha256,
            'd_tri_vs_edge_rgb': str(edge_path),
            'd_vert_vs_vertex_rgb': str(vert_path),
        })

    unload_flow(gaussian_flow)

summary_path = OUTPUT_DIR / 'summary.json'
with open(summary_path, 'w') as fp:
    json.dump(summary, fp, indent=2)
print('Saved summary:', summary_path)
print('Saved images:', len(summary) * 2)

Loaded gaussian_shape_kl5e3: Gaussian KL5e-3 + shape latent
  checkpoint: /nfs/turbo/coe-jjparkcv-medium/koussa/neuframe/outputs/michelangelo_shape2gaussian_distance_flow_kl5e3/ckpts/denoiser_step0060000.pt
[gaussian_shape_kl5e3] [1/63] 0083b931abca7ba79abbb9e7ef851633297d0860436fd677d2279927867bbd6d
[gaussian_shape_kl5e3] [2/63] 020452235d4c2b085455c6d34afa86b15c0d9f737bda5fb90d944dc0f285e640
[gaussian_shape_kl5e3] [3/63] 035e8e5f3fb4f1103d1ab67456283f4d038a8537f5d25454843594fdd90d1f53
[gaussian_shape_kl5e3] [4/63] 049db641b0942cfca4f3bc0039a913276c21bc9e42cecf52e1a971b15d9e9763
[gaussian_shape_kl5e3] [5/63] 0b008fcdc842a59acee79b24a104cd688b0669ca9392d296c28c83f12fc73e0b
[gaussian_shape_kl5e3] [6/63] 0b7ce9b1d6bb838deb7936aae55dbbedc045c0bdb8f7b2363d2bcc38dc95b0dd
[gaussian_shape_kl5e3] [7/63] 0e49c47a6e29c2333c35e4741f30a9920904d147b3919213a46747e0f3c79292
[gaussian_shape_kl5e3] [8/63] 1283d3ddfcd2a7917b780c3200aa7c64574ed5682c2719b42fe6981a0daa2ba5
[gaussian_shape_kl5e3] [9/63] 1a0

## Preview a Saved Pair

In [ ]:
if summary:
    display(Image.open(summary[0]['d_tri_vs_edge_rgb']))
    display(Image.open(summary[0]['d_vert_vs_vertex_rgb']))
else:
    print('Run the comparison cell first.')